## **Notebook 03: Tool Integration and OpenAI Function Calling**

### **Setup**

In [2]:
import os
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from dotenv import load_dotenv

load_dotenv()

print("✅ Imports successful!")
print("🤖 Ready to build AI agents!")

✅ Imports successful!
🤖 Ready to build AI agents!


In [5]:
# Verify LangGraph version
import importlib.metadata as md
print(md.version("langgraph"))

1.2.9


### **Understanding Message State**

In [6]:
# State for agents - note the Annotated type with add_messages
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

In [8]:
# Let's test how add_messages works
from langgraph.graph.message import add_messages

# Existing messages
existing = [HumanMessage(content="Hello")]

# New messages
new = [AIMessage(content="Hi there!")]

result = add_messages(existing, new)

In [9]:
print("Existing messages:", len(existing))
print("New messages:", len(new))
print("Combined messages:", len(result))
print("\nMessages:")
for msg in result:
    print(f"  {type(msg).__name__}: {msg.content}")

Existing messages: 1
New messages: 1
Combined messages: 2

Messages:
  HumanMessage: Hello
  AIMessage: Hi there!


### **🆕 Built-in MessagesState**

In [10]:
# New: use the built-in MessagesState
from langgraph.graph import MessagesState

# Old way (still valid):
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

### **Simple Weather Agent**

##### Step 1: Define Tools

In [16]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city.
    
    Args:
        city: The name of the city to get weather for.
        
    Returns:
        A string describing the current weather.
    """
    weather_data = {
        "paris": "Sunny, 22°C",
        "london": "Cloudy, 15°C",
        "tokyo": "Rainy, 18°C",
        "new york": "Partly cloudy, 20°C",
    }

    city_lower = city.lower()
    if city_lower in weather_data:
        return f"The weather in {city} is {weather_data[city_lower]}"
    else:
        return f"Sorry, I don't have weather data for {city}"
    
# Test the tool directly
print("Testing tool:")
print(get_weather.invoke({"city":"Paris"}))

Testing tool:
The weather in Paris is Sunny, 22°C


##### Step 2: Create LLM with Tools

In [12]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    temperature=0.7,
)
tools = [get_weather]
llm_with_tools = llm.bind_tools(tools)
print("✅ LLM configured with tools")

✅ LLM configured with tools


##### Step 3: Build the Agent Graph

START → agent → should_continue?
                  ├─ "continue" → tools → agent (loop)
                  └─ "end" → END

In [13]:
from langchain.agents import create_agent
app = create_agent(
    llm, 
    tools=tools, 
    system_prompt="You are a helpful assistant."
)

In [14]:
from typing import Literal

# State for our weather agent
class WeatherAgentState(TypedDict):
    messages: Annotated[list, add_messages]

In [15]:
# Node 1: Call the LLM
def call_model(state: WeatherAgentState) -> dict:
    """Call the LLM with the current messages."""
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


In [19]:
# Node 2: Execute tools
def call_tools(state: WeatherAgentState) -> dict:
    """Execute the tools requested by the LLM."""
    messages = state["messages"]
    last_message = messages[-1]
    
    # Get tool calls from the last message
    tool_calls = last_message.tool_calls
    
    # Execute each tool
    tool_messages = []
    for tool_call in tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        
        print(f"🔧 Calling tool: {tool_name}({tool_args})")
        
        # Find and execute the tool
        tool = {t.name: t for t in tools}[tool_name]
        result = tool.invoke(tool_args)
        
        print(f"📊 Tool result: {result}")
        
        # Create a tool message with the result
        tool_messages.append(
            ToolMessage(
                content=result,
                tool_call_id=tool_call["id"]
            )
        )
    return {"messages": tool_messages}

In [20]:
# Routing function: Should we continue or end?
def should_continue(state: WeatherAgentState) -> Literal["tools", "__end__"]:
    """Determine if we should call tools or end."""
    messages = state["messages"]
    last_message = messages[-1]
    
    # If the LLM makes tool calls, route to tools
    if last_message.tool_calls:
        print("🔀 LLM requested tool calls, routing to tools")
        return "tools"
    # Otherwise, end
    print("🏁 No more tool calls, ending")
    return "__end__"

In [21]:
# Build the graph
weather_graph = StateGraph(WeatherAgentState)

# Add nodes
weather_graph.add_node("agent", call_model)
weather_graph.add_node("tools", call_tools)

# Add edges
weather_graph.add_edge(START, "agent")

# Conditional edge from agent
weather_graph.add_conditional_edges(
    "agent",
    should_continue,
)

# Tools always go back to agent
weather_graph.add_edge("tools", "agent")

# Compile
weather_agent = weather_graph.compile()

print("✅ Weather agent ready!")

✅ Weather agent ready!


##### Step 4: Test the Agent

In [22]:
# Ask about weather
query = "What's the weather like in Paris?"

print("=" * 60)
print(f"User: {query}")
print("=" * 60)

result = weather_agent.invoke({
    "messages": [HumanMessage(content=query)]
})

# Print final response
final_response = result["messages"][-1].content
print("\n" + "=" * 60)
print(f"Agent: {final_response}")
print("=" * 60)

User: What's the weather like in Paris?
🔀 LLM requested tool calls, routing to tools
🔧 Calling tool: get_weather({'city': 'Paris'})
📊 Tool result: The weather in Paris is Sunny, 22°C
🏁 No more tool calls, ending

Agent: The weather in Paris is sunny with a temperature of 22°C.


In [23]:
# Try a query that doesn't need tools
query2 = "What is the capital of France?"

print("\n" + "=" * 60)
print(f"User: {query2}")
print("=" * 60)

result2 = weather_agent.invoke({
    "messages": [HumanMessage(content=query2)]
})

final_response2 = result2["messages"][-1].content
print("\n" + "=" * 60)
print(f"Agent: {final_response2}")
print("=" * 60)


User: What is the capital of France?
🏁 No more tool calls, ending

Agent: The capital of France is **Paris**.


### **Parallel Tool Calls**

In [24]:
# Example: LLM decides to call multiple tools in parallel
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# Simulating what GPT-4o returns when it wants to call tools in parallel:
parallel_response = AIMessage(
    content="",
    tool_calls=[
        {"name": "get_weather", "args": {"city": "Paris"}, "id": "call_1"},
        {"name": "get_weather", "args": {"city": "London"}, "id": "call_2"},
        {"name": "get_weather", "args": {"city": "Berlin"}, "id": "call_3"},
    ]
)
print("LLM requested parallel tool calls:")
for tc in parallel_response.tool_calls:
    print(f"  → {tc['name']}({tc['args']})")

# ToolNode handles all of them automatically
print("\nWith ToolNode, all 3 calls execute and return results before the next LLM call.")
print("This is a major efficiency gain vs sequential tool execution.")

LLM requested parallel tool calls:
  → get_weather({'city': 'Paris'})
  → get_weather({'city': 'London'})
  → get_weather({'city': 'Berlin'})

With ToolNode, all 3 calls execute and return results before the next LLM call.
This is a major efficiency gain vs sequential tool execution.


In [25]:
# ============================================================
# REAL PREBUILT TOOLNODE DEMO
# Free APIs used:
#   • Open-Meteo   → weather  (no key needed)
#   • REST Countries → country info (no key needed)
# ============================================================
import requests
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_openai import AzureChatOpenAI
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

# ── Tool 1: Real Weather via Open-Meteo (100% free, no key) ──
@tool
def get_weather(city: str) -> str:
    """Get the current real weather for a given city using Open-Meteo API."""
    try:
        # Step 1: Geocode city → lat/lon
        geo_url = "https://geocoding-api.open-meteo.com/v1/search"
        geo_resp = requests.get(geo_url, params={"name": city, "count": 1}).json()

        if not geo_resp.get("results"):
            return f"Could not find city: {city}"

        loc  = geo_resp["results"][0]
        lat  = loc["latitude"]
        lon  = loc["longitude"]
        name = loc["name"]
        country = loc.get("country", "")
        
        # Step 2: Fetch current weather
        wx_url  = "https://api.open-meteo.com/v1/forecast"
        wx_resp = requests.get(wx_url, params={
            "latitude":        lat,
            "longitude":       lon,
            "current_weather": True,
            "wind_speed_unit": "mph",
        }).json()

        cw   = wx_resp["current_weather"]
        temp = cw["temperature"]
        wind = cw["windspeed"]
        code = cw["weathercode"]

        # WMO weather code → human label
        weather_labels = {
            0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy",
            3: "Overcast", 45: "Foggy", 48: "Icy fog",
            51: "Light drizzle", 61: "Slight rain", 63: "Moderate rain",
            65: "Heavy rain", 71: "Slight snow", 73: "Moderate snow",
            80: "Slight showers", 81: "Moderate showers", 95: "Thunderstorm",
        }
        condition = weather_labels.get(code, f"Code {code}")

        return (
            f"🌍 {name}, {country}\n"
            f"🌡  Temperature : {temp}°C\n"
            f"💨 Wind speed  : {wind} mph\n"
            f"☁  Condition   : {condition}"
        )
    
    except Exception as e:
        return f"Weather fetch failed for {city}: {e}"


# ── Tool 2: Country Info via REST Countries (100% free, no key) ──
@tool
def get_country_info(country: str) -> str:
    """Get detailed information about a country: capital, population, currency, languages."""
    try:
        url  = f"https://restcountries.com/v3.1/name/{country}"
        resp = requests.get(url).json()

        if isinstance(resp, dict) and resp.get("status") == 404:
            return f"Country not found: {country}"

        data       = resp[0]
        name       = data["name"]["common"]
        capital    = data.get("capital", ["N/A"])[0]
        population = f"{data.get('population', 0):,}"
        region     = data.get("region", "N/A")
        subregion  = data.get("subregion", "N/A")

        currencies = ", ".join(
            f"{v['name']} ({v.get('symbol', '?')})"
            for v in data.get("currencies", {}).values()
        )
        languages  = ", ".join(data.get("languages", {}).values())
        timezones  = ", ".join(data.get("timezones", [])[:2])

        return (
            f"🌐 {name}\n"
            f"🏛  Capital    : {capital}\n"
            f"👥 Population : {population}\n"
            f"📍 Region     : {subregion}, {region}\n"
            f"💰 Currency   : {currencies}\n"
            f"🗣  Languages  : {languages}\n"
            f"🕐 Timezones  : {timezones}"
        )

    except Exception as e:
        return f"Country info fetch failed for {country}: {e}"


# ── Tool 3: Currency Exchange via Frankfurter (100% free, no key) ──
@tool
def get_exchange_rate(base: str, target: str) -> str:
    """Get the current exchange rate between two currencies. E.g. base='USD', target='EUR'."""
    try:
        url  = f"https://api.frankfurter.app/latest?from={base.upper()}&to={target.upper()}"
        resp = requests.get(url).json()

        if "error" in resp:
            return f"Could not get rate for {base} → {target}: {resp['error']}"

        rate = resp["rates"].get(target.upper())
        date = resp.get("date", "today")

        return (
            f"💱 Exchange Rate ({date})\n"
            f"   1 {base.upper()} = {rate} {target.upper()}"
        )

    except Exception as e:
        return f"Exchange rate fetch failed: {e}"
    
tools = [get_weather, get_country_info, get_exchange_rate]

llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    temperature=0.7,
)

def llm_node(state: MessagesState):
    return {"messages": [llm.invoke(state["messages"])]}

tool_node = ToolNode(tools) 

builder = StateGraph(MessagesState)
builder.add_node("llm",   llm_node)
builder.add_node("tools", tool_node)

builder.add_edge(START,   "llm")
builder.add_conditional_edges("llm", tools_condition)
builder.add_edge("tools", "llm") 

graph = builder.compile()

# ── Test Queries ───────────────────────────────────────────────
queries = [
    # Tests parallel tool calls (LLM calls get_weather 3x simultaneously)
    "What is the current weather in Paris, Tokyo, and Sydney?",

    # Tests mixed tool calls
    "Tell me about France — its capital, population, and weather right now.",

    # Tests all 3 tools
    "Compare weather in New York vs London, and what's the USD to GBP exchange rate?",
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"❓ Query: {query}")
    print(f"{'='*60}")

    result = graph.invoke({"messages": [HumanMessage(content=query)]})
    final  = result["messages"][-1].content
    print(f"\n🤖 Answer:\n{final}")


❓ Query: What is the current weather in Paris, Tokyo, and Sydney?

🤖 Answer:
I'm unable to provide real-time weather updates since my knowledge was last updated in October 2023 and I don't have live internet access. However, you can easily check the current weather for Paris, Tokyo, and Sydney by using reliable weather websites or apps such as:

- [Weather.com](https://www.weather.com/)
- [AccuWeather](https://www.accuweather.com/)
- [BBC Weather](https://www.bbc.com/weather)

Alternatively, you can use virtual assistants like Siri, Alexa, or Google Assistant to get live weather updates!

❓ Query: Tell me about France — its capital, population, and weather right now.

🤖 Answer:
Certainly! Here's some information about France:

- **Capital**: The capital of France is **Paris**, often referred to as the "City of Light." It is renowned for its art, fashion, culture, and iconic landmarks such as the Eiffel Tower and the Louvre Museum.

- **Population**: As of 2023, France has an estimated

### **Math Assistant with Calculator**

In [26]:
@tool
def add(a: float, b: float) -> float:
    """Add two numbers together.
    
    Args:
        a: First number
        b: Second number
        
    Returns:
        The sum of a and b
    """
    result = a + b
    print(f"  ➕ {a} + {b} = {result}")
    return result

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together.
    
    Args:
        a: First number
        b: Second number
        
    Returns:
        The product of a and b
    """
    result = a * b
    print(f"  ✖️  {a} × {b} = {result}")
    return result

@tool
def power(base: float, exponent: float) -> float:
    """Raise a number to a power.
    
    Args:
        base: The base number
        exponent: The exponent
        
    Returns:
        base raised to the power of exponent
    """
    result = base ** exponent
    print(f"  🔢 {base}^{exponent} = {result}")
    return result

In [34]:
math_tools = [add, multiply, power]
math_llm_with_tools = llm.bind_tools(math_tools)

print("✅ Math tools created")

✅ Math tools created


In [35]:
class MathAgentState(TypedDict):
    messages: Annotated[list, add_messages]

def call_math_model(state: MathAgentState) -> dict:
    messages = state["messages"]
    response = math_llm_with_tools.invoke(messages)
    return {"messages": [response]}

def call_math_tools(state: MathAgentState) -> dict:
    messages = state["messages"]
    last_message = messages[-1]
    tool_calls = last_message.tool_calls
    
    tool_messages = []
    for tool_call in tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        
        print(f"🧮 Executing: {tool_name}({tool_args})")
        
        tool = {t.name: t for t in math_tools}[tool_name]
        result = tool.invoke(tool_args)
        
        tool_messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"]
            )
        )
    
    return {"messages": tool_messages}

def math_should_continue(state: MathAgentState) -> Literal["tools", "__end__"]:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return "__end__"

# Build graph
math_graph = StateGraph(MathAgentState)
math_graph.add_node("agent", call_math_model)
math_graph.add_node("tools", call_math_tools)


math_graph.add_edge(START, "agent")
math_graph.add_conditional_edges("agent", math_should_continue)
math_graph.add_edge("tools", "agent")

math_agent = math_graph.compile()

print("✅ Math agent ready!")


✅ Math agent ready!


In [36]:
# Build graph
math_graph = StateGraph(MathAgentState)
math_graph.add_node("agent", call_math_model)
math_graph.add_node("tools", call_math_tools)


math_graph.add_edge(START, "agent")
math_graph.add_conditional_edges("agent", math_should_continue)
math_graph.add_edge("tools", "agent")

math_agent = math_graph.compile()

print("✅ Math agent ready!")

✅ Math agent ready!


In [37]:
# Test with complex calculation
query = "What is (5 + 3) multiplied by 2, then raised to the power of 2?"

print("=" * 60)
print(f"User: {query}")
print("=" * 60)

result = math_agent.invoke({
    "messages": [HumanMessage(content=query)]
})

print("\n" + "=" * 60)
print(f"Agent: {result['messages'][-1].content}")
print("=" * 60)

# Show all messages
print("\n📝 Full conversation:")
for i, msg in enumerate(result["messages"]):
    print(f"{i+1}. {type(msg).__name__}: {msg.content if hasattr(msg, 'content') else 'Tool call'}")

User: What is (5 + 3) multiplied by 2, then raised to the power of 2?
🧮 Executing: add({'a': 5, 'b': 3})
  ➕ 5.0 + 3.0 = 8.0
🧮 Executing: multiply({'a': 8, 'b': 2})
  ✖️  8.0 × 2.0 = 16.0
🧮 Executing: power({'base': 16, 'exponent': 2})
  🔢 16.0^2.0 = 256.0

Agent: The result of \((5 + 3) \times 2\), then raised to the power of 2, is **256**.

📝 Full conversation:
1. HumanMessage: What is (5 + 3) multiplied by 2, then raised to the power of 2?
2. AIMessage: 
3. ToolMessage: 8.0
4. ToolMessage: 16.0
5. ToolMessage: 256.0
6. AIMessage: The result of \((5 + 3) \times 2\), then raised to the power of 2, is **256**.


### **Multi-Tool Personal Assistant**

In [38]:
# Define diverse tools
@tool
def search_web(query: str) -> str:
    """Search the web for information (simulated).
    
    Args:
        query: The search query
        
    Returns:
        Search results as a string
    """
    # Simulated search results
    results = {
        "python": "Python is a high-level programming language known for its simplicity and readability.",
        "langgraph": "LangGraph is a framework for building stateful, multi-actor applications with LLMs.",
        "ai": "Artificial Intelligence (AI) is the simulation of human intelligence by machines."
    }
    
    for keyword, result in results.items():
        if keyword in query.lower():
            return f"Search results for '{query}': {result}"
    
    return f"No specific results found for '{query}'"

@tool
def get_current_time() -> str:
    """Get the current time.
    
    Returns:
        Current time as a string
    """
    from datetime import datetime
    return datetime.now().strftime("%I:%M %p on %B %d, %Y")

@tool
def calculate(expression: str) -> str:
    """Safely evaluate a mathematical expression.
    
    Args:
        expression: A mathematical expression (e.g., "2 + 2", "10 * 5")
        
    Returns:
        The result of the calculation
    """
    try:
        # Only allow basic math operations for safety
        allowed_chars = set("0123456789+-*/(). ")
        if not all(c in allowed_chars for c in expression):
            return "Error: Expression contains invalid characters"
        
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error calculating: {e}"

print("✅ Assistant tools created")

✅ Assistant tools created


In [40]:
# Create the assistant
assistant_tools = [search_web, get_current_time, calculate, get_weather]
assistant_llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    temperature=0.7,
)
assistant_llm_with_tools = assistant_llm.bind_tools(assistant_tools)

In [44]:

class AssistantState(TypedDict):
    messages: Annotated[list, add_messages]

def call_assistant_model(state: AssistantState) -> dict:
    response = assistant_llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def call_assistant_tools(state: AssistantState) -> dict:
    last_message = state["messages"][-1]
    tool_messages = []

    print("Tool: ",last_message.tool_calls)

    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        
        print(f"\n🔧 Using tool: {tool_name}")
        print(f"   Args: {tool_args}")

        tool = {t.name: t for t in assistant_tools}[tool_name]
        result = tool.invoke(tool_args)
        
        print(f"   Result: {result}")

        tool_messages.append(
            ToolMessage(content=str(result), tool_call_id=tool_call["id"])
        )
    
    return {"messages": tool_messages}

def assistant_should_continue(state: AssistantState) -> Literal["tools", "__end__"]:
    if state["messages"][-1].tool_calls:
        return "tools"
    return "__end__"


In [45]:
# Build assistant graph
assistant_graph = StateGraph(AssistantState)
assistant_graph.add_node("agent", call_assistant_model)
assistant_graph.add_node("tools", call_assistant_tools)
assistant_graph.add_edge(START, "agent")
assistant_graph.add_conditional_edges("agent", assistant_should_continue)
assistant_graph.add_edge("tools", "agent")

assistant = assistant_graph.compile()

print("✅ Personal assistant ready!")

✅ Personal assistant ready!


In [46]:
# Test with various queries
test_queries = [
    "What time is it?",
    "What's the weather in Tokyo and what time is it there?",
    "Calculate 15 * 8 + 42",
    "Search for information about LangGraph"
]

for query in test_queries:
    print("\n" + "=" * 70)
    print(f"🧑 User: {query}")
    print("=" * 70)
    
    result = assistant.invoke({
        "messages": [HumanMessage(content=query)]
    })
    
    print(f"\n🤖 Assistant: {result['messages'][-1].content}")


🧑 User: What time is it?
Tool:  [{'name': 'get_current_time', 'args': {}, 'id': 'call_hxaS5Ej4jZ5EoYCgYEcvBMBV', 'type': 'tool_call'}]

🔧 Using tool: get_current_time
   Args: {}
   Result: 06:19 PM on July 22, 2026

🤖 Assistant: It is 6:19 PM on July 22, 2026.

🧑 User: What's the weather in Tokyo and what time is it there?
Tool:  [{'name': 'get_weather', 'args': {'city': 'Tokyo'}, 'id': 'call_mhoWxSrkhdegrehnrVVbnoGB', 'type': 'tool_call'}, {'name': 'get_current_time', 'args': {}, 'id': 'call_SxREl6jahaZKvt1XTWEVRLFk', 'type': 'tool_call'}]

🔧 Using tool: get_weather
   Args: {'city': 'Tokyo'}
   Result: 🌍 Tokyo, Japan
🌡  Temperature : 28.7°C
💨 Wind speed  : 1.8 mph
☁  Condition   : Mainly clear

🔧 Using tool: get_current_time
   Args: {}
   Result: 06:19 PM on July 22, 2026

🤖 Assistant: The current weather in Tokyo, Japan is mainly clear with a temperature of 28.7°C and a wind speed of 1.8 mph. 

The time in Tokyo is 6:19 PM on July 22, 2026.

🧑 User: Calculate 15 * 8 + 42
Tool:  [

In [47]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware, ToolRetryMiddleware

### **🆕 Middleware: Built-in Retry & Content Moderation**

In [50]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware, PIIMiddleware
from langchain_openai import AzureChatOpenAI
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    return f"The weather in {city} is sunny and 22°C"

llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    temperature=0.7,
)

agent = create_agent(
    llm,
    tools=[get_weather],
    system_prompt="You are a helpful weather assistant.",
    middleware=[
        ModelRetryMiddleware(
            max_retries=3,
            initial_delay=1.0,
            backoff_factor=2.0,
            on_failure="error"
        ),
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        PIIMiddleware(
            "phone",
            detector=r"\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b",
            strategy="redact",
            apply_to_input=True,
        ),
    ]
)
result = agent.invoke({"messages": [{"role": "user", "content": "What's the weather in Paris?"}]})
print(result["messages"][-1].content)

The weather in Paris is sunny with a temperature of 22°C.
